In [29]:
import pandas as pd
density = pd.read_csv('data/updated_densitymap_hourly_cleaned.csv')
bot_fill = pd.read_csv('data/bottle_fillers.csv')
bot_fill_cord = pd.read_csv('data/bottle_filler_coordinates.csv')

In [30]:
# Remove addresses that are nan
density = density.dropna(subset=['latitude', 'longitude'])
bottle_fill = bot_fill.dropna(subset=['Address A', 'Address B'])

In [66]:
density.head()

,date,hour,building_floor,building_desc,latitude,longitude,building_type,density_cnt,capacity,dc_ratio
0,2021-03-31,1,4,209 BAY STATE ROAD,42.3507,-71.1028,Residential,1,8,0.13
1,2021-03-31,1,2,200-202 BAY STATE ROAD,42.3503,-71.1026,Residential,1,12,0.08
2,2021-03-31,1,2,184 BAY STATE ROAD,42.3503,-71.1019,Residential,1,6,0.17
3,2021-03-31,1,3,190 BAY STATE ROAD,42.3503,-71.1021,Residential,1,9,0.11
4,2021-03-31,1,3,161 BAY STATE ROAD,42.3506,-71.1009,Residential,1,5,0.20


In [74]:
# Get average capacity and dc_ratio for all time at a given location
average_counts = density.groupby(['building_desc'])[['latitude', 'longitude', 'capacity', 'dc_ratio']].mean().reset_index()
average_counts.head()

,building_desc,latitude,longitude,capacity,dc_ratio
0,1 BUSWELL STREET,42.3480,-71.1044,3.755505,0.661792
1,1 UNIVERSITY ROAD,42.3512,-71.1100,162.754299,0.116412
2,10 BUICK STREET,42.3524,-71.1160,54.114409,0.275676
3,10 COTTAGE FARM ROAD,42.3480,-71.1131,0.000000,0.000000
4,10 LENOX STREET,42.3487,-71.1099,32.831234,0.180870


In [75]:
from utils import haversine

def find_clossest_water_fountain_dist(lat, lon, locations):
    p1 = (lat, lon)
    distances = [haversine(p1, location) for location in locations]
    return min(distances)

In [76]:
bot_fill_cord.head()
latitudes = bot_fill_cord.loc[:, 'latitudes'].tolist()
longitudes = bot_fill_cord.loc[:, 'longitudes'].tolist()
locations = list(zip(latitudes, longitudes))


apply_clossest_dist = lambda row: find_clossest_water_fountain_dist(row['latitude'], row['longitude'], locations)
average_counts['nearest_filler'] = average_counts.apply(apply_clossest_dist, axis=1)
average_counts['score'] = average_counts['capacity'] * average_counts['nearest_filler']
average_counts.head()

,building_desc,latitude,longitude,capacity,dc_ratio,nearest_filler,score
0,1 BUSWELL STREET,42.3480,-71.1044,3.755505,0.661792,0.101530,0.381296
1,1 UNIVERSITY ROAD,42.3512,-71.1100,162.754299,0.116412,0.031135,5.067426
2,10 BUICK STREET,42.3524,-71.1160,54.114409,0.275676,0.084101,4.551095
3,10 COTTAGE FARM ROAD,42.3480,-71.1131,0.000000,0.000000,0.275083,0.000000
4,10 LENOX STREET,42.3487,-71.1099,32.831234,0.180870,0.201382,6.611626


In [77]:
## Compute the top k locations with high capacity * distance to nearest filter
k = 10

top_locations = average_counts.sort_values(by='score', ascending=False)
top_locations.head(k)

,building_desc,latitude,longitude,capacity,dc_ratio,nearest_filler,score
111,285 BABCOCK STREET,42.3540,-71.1203,432.659502,0.109147,0.065244,28.228535
5,100 ASHFORD STREET,42.3546,-71.1236,416.953830,0.191605,0.055380,23.090943
231,930-940 COMMONWEALTH AVENUE,42.3507,-71.1178,304.149029,0.079632,0.054455,16.562375
157,590 COMMONWEALTH AVENUE,42.3484,-71.1002,255.073594,0.072161,0.060126,15.336478
224,900 COMMONWEALTH AVENUE,42.3506,-71.1162,205.266345,0.098901,0.073973,15.184100
194,775 COMMONWEALTH AVENUE,42.3509,-71.1090,224.732337,0.076220,0.063775,14.332208
191,750 COMMONWEALTH AVENUE,42.3498,-71.1080,100.001239,0.084778,0.143225,14.322680
166,619 MEMORIAL DRIVE,42.3532,-71.1078,51.471225,0.211484,0.229131,11.793666
37,15 ST. MARY'S STREET,42.3494,-71.1075,85.573792,0.036652,0.116412,9.961842
214,855 COMMONWEALTH AVENUE,42.3514,-71.1139,156.176183,0.096787,0.062978,9.835665


### This Code Script was run once to compute the addresses to Long Lat

In [19]:
import re

def average_address_if_range(address):
    ''' Takes in a range of addresses and returns the average address
        Ex: 20 - 30 Buswell St -> 25 Buswell St
    '''
    regex = r"(\d+)\s*-\s*(\d+)(.*)"
    parts = re.fullmatch(regex, address)
    if parts == None:
        return address

    start = int(parts.group(1))
    end = int(parts.group(2))
    street = parts.group(3)

    new = (start + end) // 2
    return str(new) + street

In [20]:
from functools import reduce

density_addresses = set(df.iloc[:, 3])
bottle_fill_address = set(bottle_fill['Address A'] + ' ' + bottle_fill['Address B'])

apply_transforms = lambda addresses, fns: [reduce(lambda acc, fn: fn(acc), fns, adr) for adr in addresses]

append_city = lambda address: address + ", Boston, MA"
transforms = [average_address_if_range, append_city]

density_addresses = apply_transforms(density_addresses, transforms)
bottle_fill_address = apply_transforms(bottle_fill_address, transforms)

density_addresses = list(set(density_addresses))
bottle_fill_address = list(set(bottle_fill_address))

print(density_addresses)
print(bottle_fill_address)


['700 COMMONWEALTH AVENUE, Boston, MA', '888 COMMONWEALTH AVENUE, Boston, MA', '214 BAY STATE ROAD, Boston, MA', '98 MOUNTFORT STREET, Boston, MA', '610 COMMONWEALTH AVENUE, Boston, MA', '838 COMMONWEALTH AVENUE, Boston, MA', '610 BEACON STREET, Boston, MA', '975 COMMONWEALTH AVENUE, Boston, MA', '50 WORTHINGTON ROAD, Boston, MA', '38 BUSWELL STREET, Boston, MA', '519 BEACON STREET, Boston, MA', "8 ST. MARY'S STREET, Boston, MA", '157 BAY STATE ROAD, Boston, MA', '175 BAY STATE ROAD, Boston, MA', '78 BAY STATE ROAD, Boston, MA', '67 BAY STATE ROAD, Boston, MA', '595 COMMONWEALTH AVENUE, Boston, MA', '171 AMORY STREET, Boston, MA', '180 BAY STATE ROAD, Boston, MA', '206 BAY STATE ROAD, Boston, MA', '3 BUSWELL STREET, Boston, MA', '178 BAY STATE ROAD, Boston, MA', '116 BAY STATE ROAD, Boston, MA', '48 BUSWELL STREET, Boston, MA', '75 CUMMINGTON MALL, Boston, MA', '41 BUSWELL STREET, Boston, MA', '153 BAY STATE ROAD, Boston, MA', '2 SILBER WAY, Boston, MA', '55 BUSWELL STREET, Boston, MA'

In [27]:
# Compute Longitude and latitude addresses
from geoapify import get_long_lat 

# Don't Run This code again

# longitudes = []
# latitudes = []

# for address in bottle_fill_address:
#     lat, lon = get_long_lat(address)
#     longitudes += [lon]
#     latitudes += [lat]


In [23]:
bottle_fill_coords_data = {
    'address': bottle_fill_address,
    'latitudes': latitudes,
    'longitudes': longitudes
}

bottle_fill_cords = pd.DataFrame(bottle_fill_coords_data)
bottle_fill_cords.head()
bottle_fill_cords.to_csv('data/bottle_filler_coordinates.csv')

In [ ]:
## Next, we want to compute the distances between each address

